# LISP in Python: vignette and validation

Development notebook for the project. Part 1 runs the localsp vignette workflow in Python (variogram, threshold, LISP on all 908 locations). Part 2 compares the Python output with the R reference output on the 100-row subset.

To run it in Google Colab, first run `!git clone https://github.com/gurpreetk0ur/lisp-python.git` in a cell, then run the cells below. To run it locally, set `REPO` to the folder that contains `lisp5.py`.

Set the repository path

In [ ]:
REPO = "/content/lisp-python"   # folder containing lisp5.py and data/

Handle Packages

In [ ]:
!pip install scikit-gstat geopandas --quiet

In [ ]:
import os

In [ ]:
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
import skgstat as skg
import matplotlib.pyplot as plt

In [ ]:
pd.set_option('display.max_columns', None)   # show all columns
pd.set_option('display.max_rows', 10)        # only show 10 rows (default is 60)
pd.set_option('display.width', None)         # don't wrap columns to new lines
pd.set_option('display.float_format', '{:.4f}'.format)  # tidy up float display

# Part 1: Python version of the localsp vignette

1. Import lisp5

In [ ]:
import sys
sys.path.insert(0, REPO)
from lisp5 import lisp

2. Load Data

In [ ]:
# In the R vignette:
#   gtc = readr::read_csv(system.file("extdata/gtc.csv", package = "localsp"))
gtc_raw = pd.read_csv(f"{REPO}/data/gtc.csv")
print("Raw data shape:", gtc_raw.shape)
gtc_raw.head()

3. Build GeoDataframe

In [ ]:
# R: gtc = sf::st_as_sf(gtc, coords = c("X","Y"), crs = 4326)
gtc = gpd.GeoDataFrame(
    gtc_raw.drop(columns=['X', 'Y']),
    geometry=gpd.points_from_xy(gtc_raw['X'], gtc_raw['Y']),
    crs='EPSG:4326',
)
gtc.head()

Fit Variogram:

In [ ]:
coords = np.column_stack([gtc_raw['X'].values, gtc_raw['Y'].values])
values = gtc_raw['GTC'].values
values_detrended = values - values.mean()  # mirrors GTC ~ 1 in R

best_v, best_rmse = None, np.inf

for model_name in ['spherical', 'exponential', 'gaussian', 'matern']:
    try:
        v = skg.Variogram(coordinates=coords, values=values_detrended,
                          model=model_name, n_lags=15, maxlag='median')
        rmse   = v.rmse
        vrange = v.parameters[0]
        print(f"  {model_name:12s}  RMSE={rmse:.6f}  range={vrange:.4f}")
        if rmse < best_rmse:
            best_rmse, best_v = rmse, v
    except Exception as e:
        print(f"  {model_name:12s}  failed: {type(e).__name__}: {e}")

if best_v is None:
    raise RuntimeError("All variogram models failed.")

vrange  = best_v.parameters[0]
print(f"\nBest model : {best_v.model.__name__}  RMSE={best_rmse:.6f}")
print(f"Range      : {vrange:.6f}")

5. Plot Variogram

In [ ]:
fig = best_v.plot()
plt.title(f"Fitted variogram: {best_v.model.__name__} (range={vrange:.3f})")
plt.tight_layout()
plt.savefig('variogram_plot.png', dpi=150)
plt.show()

6. Compute Threshold

The threshold is set to 2 × range, as in the LISP paper. The R vignette uses 4 × range.

In [ ]:
threshold = vrange * 2
print(f"Variogram range : {vrange:.4f}")
print(f"Threshold (×2)  : {threshold:.4f}")

7. Run LISP

Demonstration run on all 908 locations, using the validated settings (3 to 5 classes, quantile). With the 2 × range threshold, location 473 has only 2 neighbours and is skipped with a warning.

In [ ]:
# R: lpd = localsp::lisp(GTC ~ ., data = gtc, threshold, distmat, cores = 12)

# lisp5 takes coordinates directly, so no distance matrix is passed in
data_for_lisp = gtc_raw.drop(columns=['X', 'Y'])

lpd = lisp(
    formula='GTC ~ .',
    data=data_for_lisp,
    coords=coords,
    threshold=threshold,
    discnum=list(range(3, 6)),
    discmethod=['quantile'],
)

print(f"\nShape: {lpd.shape}")
lpd

# Part 2: Validation against the R reference output

In [ ]:
sys.path.insert(0, REPO)
from lisp5 import lisp

gtc1 = pd.read_csv(f"{REPO}/data/gtc_subset_100.csv")
r_df = pd.read_csv(f"{REPO}/data/r_ground_truth_100.csv")

Run the Python translation on the 100-row subset (same settings as the R reference)

In [ ]:
import pandas as pd

coords = gtc1[["X", "Y"]].values
data = gtc1.drop(columns=["X", "Y"]).reset_index(drop=True)

result = lisp("GTC ~ .", data=data, coords=coords, threshold=8,
              discnum=range(3, 6), discmethod=["quantile"])
result.to_csv(f"{REPO}/validation/py_result_100.csv", index=False)

print(result.shape)
result.head(10)

Diff against R ground truth

In [ ]:
py_df = pd.read_csv(f"{REPO}/validation/py_result_100.csv")
assert (r_df["rid"].values == py_df["rid"].values).all(), "rid order mismatch: sort both by rid first"

In [ ]:
print("R columns: ", sorted(r_df.columns.tolist()))
print("Py columns:", sorted(py_df.columns.tolist()))

missing_in_py = [c for c in r_df.columns if c not in py_df.columns]
missing_in_r  = [c for c in py_df.columns if c not in r_df.columns]
print("\nColumns in R but not Python:", missing_in_py)
print("Columns in Python but not R:", missing_in_r)

common_cols = [c for c in r_df.columns if c in py_df.columns and c != "rid"]
numeric_cols = [c for c in common_cols if pd.api.types.is_numeric_dtype(r_df[c])]

diff = (r_df[numeric_cols].reset_index(drop=True) -
        py_df[numeric_cols].reset_index(drop=True)).abs()

diff.describe().T[["mean", "max"]].sort_values("max", ascending=False)

In [ ]:
summary = diff.describe().T[["mean", "max"]]
worst = summary.sort_values("max", ascending=False)

print("Max difference across ALL columns:", summary["max"].max())
print("\nAny column with max diff > 1e-6:")
print(worst[worst["max"] > 1e-6])

# View Diff

In [ ]:
r_df  = pd.read_csv(f"{REPO}/data/r_ground_truth_100.csv").sort_values("rid").reset_index(drop=True)
py_df = pd.read_csv(f"{REPO}/validation/py_result_100.csv").sort_values("rid").reset_index(drop=True)

# check both are aligned on the same rows before comparing
assert (r_df["rid"].values == py_df["rid"].values).all(), "rid order mismatch: check both files"

In [ ]:
compare_cols = ["pd_Slope", "sig_Slope", "pd_Elev", "sig_Elev"]

side_by_side = pd.DataFrame({"rid": r_df["rid"].head(10)})
for col in compare_cols:
    side_by_side[f"R_{col}"] = r_df[col].head(10).values
    side_by_side[f"Py_{col}"] = py_df[col].head(10).values

display(side_by_side)